In [ ]:
import pandas as pd
import json
import sqlite3
import os

STEP 1: CREATE SAMPLE CSV FILES

In [ ]:
csv1 = pd.DataFrame({
    "Student_ID": [1, 2, 3, 4],
    "Name": ["Rahul", "Sneha", "Amit", "Priya"],
    "Age": [20, 21, 22, 20],
    "Marks": [85, 92, 78, 88]
})

csv2 = pd.DataFrame({
    "Student_ID": [5, 6, 7, 8],
    "Name": ["Rohan", "Neha", "Karan", "Anita"],
    "Age": [22, 21, 20, 23],
    "Marks": [65, 95, 82, 90]
})

csv1.to_csv("/content/students1.csv", index=False)
csv2.to_csv("/content/students2.csv", index=False)

print("CSV files created successfully.")

CSV files created successfully.


STEP 2: EXTRACT DATA FROM MULTIPLE CSV FILES

In [ ]:
df1 = pd.read_csv("/content/students1.csv")
df2 = pd.read_csv("/content/students2.csv")

print("\n--- CSV File 1 ---")
print(df1)

print("\n--- CSV File 2 ---")
print(df2)


--- CSV File 1 ---
   Student_ID   Name  Age  Marks
0           1  Rahul   20     85
1           2  Sneha   21     92
2           3   Amit   22     78
3           4  Priya   20     88

--- CSV File 2 ---
   Student_ID   Name  Age  Marks
0           5  Rohan   22     65
1           6   Neha   21     95
2           7  Karan   20     82
3           8  Anita   23     90


STEP 3: COMBINE MULTIPLE CSV FILES

In [ ]:
combined_df = pd.concat(
    [df1, df2],
    ignore_index=True
)

print("\n--- Combined Dataset ---")
print(combined_df)



--- Combined Dataset ---
   Student_ID   Name  Age  Marks
0           1  Rahul   20     85
1           2  Sneha   21     92
2           3   Amit   22     78
3           4  Priya   20     88
4           5  Rohan   22     65
5           6   Neha   21     95
6           7  Karan   20     82
7           8  Anita   23     90


STEP 4: IDENTIFY INVALID RECORDS

In [ ]:
# Invalid records:
# Age should be between 18 and 25
# Marks should be between 0 and 100

invalid_records = combined_df[
    (combined_df["Age"] < 18) |
    (combined_df["Age"] > 25) |
    (combined_df["Marks"] < 0) |
    (combined_df["Marks"] > 100)
]

print("\n--- Invalid Records ---")
print(invalid_records)


--- Invalid Records ---
Empty DataFrame
Columns: [Student_ID, Name, Age, Marks]
Index: []


STEP 5: REMOVE INVALID RECORDS

In [ ]:
cleaned_df = combined_df[
    (combined_df["Age"] >= 18) &
    (combined_df["Age"] <= 25) &
    (combined_df["Marks"] >= 0) &
    (combined_df["Marks"] <= 100)
].copy()

print("\n--- Dataset After Removing Invalid Records ---")
print(cleaned_df)


--- Dataset After Removing Invalid Records ---
   Student_ID   Name  Age  Marks
0           1  Rahul   20     85
1           2  Sneha   21     92
2           3   Amit   22     78
3           4  Priya   20     88
4           5  Rohan   22     65
5           6   Neha   21     95
6           7  Karan   20     82
7           8  Anita   23     90


STEP 6: TRANSFORM DATA

In [ ]:
# Convert names to uppercase

cleaned_df["Name"] = cleaned_df["Name"].str.upper()

# Create performance category

def performance(marks):
    if marks >= 85:
        return "Excellent"
    elif marks >= 70:
        return "Good"
    else:
        return "Needs Improvement"

cleaned_df["Performance"] = cleaned_df["Marks"].apply(
    performance
)

print("\n--- Transformed Dataset ---")
print(cleaned_df)


--- Transformed Dataset ---
   Student_ID   Name  Age  Marks        Performance
0           1  RAHUL   20     85          Excellent
1           2  SNEHA   21     92          Excellent
2           3   AMIT   22     78               Good
3           4  PRIYA   20     88          Excellent
4           5  ROHAN   22     65  Needs Improvement
5           6   NEHA   21     95          Excellent
6           7  KARAN   20     82               Good
7           8  ANITA   23     90          Excellent


STEP 7: CREATE JSON DATA

In [ ]:
json_data = [
    {
        "student_id": 1,
        "course": "Data Science",
        "city": "Mumbai"
    },
    {
        "student_id": 2,
        "course": "Data Science",
        "city": "Pune"
    },
    {
        "student_id": 3,
        "course": "Artificial Intelligence",
        "city": "Delhi"
    },
    {
        "student_id": 4,
        "course": "Data Science",
        "city": "Mumbai"
    }
]

with open("/content/students.json", "w") as file:
    json.dump(json_data, file)

print("\nJSON file created successfully.")



JSON file created successfully.


STEP 8: EXTRACT AND TRANSFORM JSON DATA

In [ ]:
with open("/content/students.json", "r") as file:
    json_records = json.load(file)

json_df = pd.DataFrame(json_records)

# Rename fields

json_df.rename(
    columns={
        "student_id": "Student_ID"
    },
    inplace=True
)

# Convert city to uppercase

json_df["city"] = json_df["city"].str.upper()

print("\n--- Transformed JSON Data ---")
print(json_df)


--- Transformed JSON Data ---
   Student_ID                   course    city
0           1             Data Science  MUMBAI
1           2             Data Science    PUNE
2           3  Artificial Intelligence   DELHI
3           4             Data Science  MUMBAI


STEP 9: CREATE RELATIONAL DATABASE

In [ ]:
database = "/content/ETL_Practical7.db"

connection = sqlite3.connect(database)

print("\nSQLite database connected successfully.")


SQLite database connected successfully.


STEP 10: DATA VALIDATION BEFORE LOADING

In [ ]:
# Check for missing values

missing_values = cleaned_df.isnull().sum()

print("\n--- Missing Values ---")
print(missing_values)


# Check for duplicate Student IDs

duplicates = cleaned_df[
    cleaned_df["Student_ID"].duplicated()
]

print("\n--- Duplicate Records ---")
print(duplicates)


# Validation condition

if (
    cleaned_df["Student_ID"].notnull().all()
    and cleaned_df["Name"].notnull().all()
    and cleaned_df["Age"].between(18, 25).all()
    and cleaned_df["Marks"].between(0, 100).all()
    and not cleaned_df["Student_ID"].duplicated().any()
):

    print("\nData validation successful.")
    print("Data is ready for database loading.")

else:

    print("\nData validation failed.")
    print("Data will not be loaded.")


--- Missing Values ---
Student_ID     0
Name           0
Age            0
Marks          0
Performance    0
dtype: int64

--- Duplicate Records ---
Empty DataFrame
Columns: [Student_ID, Name, Age, Marks, Performance]
Index: []

Data validation successful.
Data is ready for database loading.


STEP 11: LOAD CSV DATA INTO DATABASE

In [ ]:
cleaned_df.to_sql(
    "Students",
    connection,
    if_exists="replace",
    index=False
)

print("\nCSV data loaded into Students table.")


CSV data loaded into Students table.


 STEP 12: LOAD JSON DATA INTO DATABASE

In [ ]:
json_df.to_sql(
    "Student_Courses",
    connection,
    if_exists="replace",
    index=False
)

print("JSON data loaded into Student_Courses table.")

JSON data loaded into Student_Courses table.


STEP 13: VERIFY DATABASE DATA

In [ ]:
print("\n--- Students Table ---")

result = pd.read_sql(
    "SELECT * FROM Students",
    connection
)

print(result)


print("\n--- Student Courses Table ---")

result_json = pd.read_sql(
    "SELECT * FROM Student_Courses",
    connection
)

print(result_json)


--- Students Table ---
   Student_ID   Name  Age  Marks        Performance
0           1  RAHUL   20     85          Excellent
1           2  SNEHA   21     92          Excellent
2           3   AMIT   22     78               Good
3           4  PRIYA   20     88          Excellent
4           5  ROHAN   22     65  Needs Improvement
5           6   NEHA   21     95          Excellent
6           7  KARAN   20     82               Good
7           8  ANITA   23     90          Excellent

--- Student Courses Table ---
   Student_ID                   course    city
0           1             Data Science  MUMBAI
1           2             Data Science    PUNE
2           3  Artificial Intelligence   DELHI
3           4             Data Science  MUMBAI


STEP 14: INCREMENTAL DATA LOADING

In [ ]:
# New records arriving later

new_data = pd.DataFrame({
    "Student_ID": [9, 10],
    "Name": ["Vikas", "Pooja"],
    "Age": [21, 22],
    "Marks": [87, 91],
    "Performance": ["Excellent", "Excellent"]
})

print("\n--- New Incoming Records ---")
print(new_data)


# Get existing Student IDs

existing_ids = pd.read_sql(
    "SELECT Student_ID FROM Students",
    connection
)["Student_ID"].tolist()


# Select only new records

incremental_data = new_data[
    ~new_data["Student_ID"].isin(existing_ids)
]


print("\n--- Records Selected for Incremental Loading ---")
print(incremental_data)


# Load only new records

if not incremental_data.empty:

    incremental_data.to_sql(
        "Students",
        connection,
        if_exists="append",
        index=False
    )

    print("\nIncremental loading completed successfully.")

else:

    print("\nNo new records found.")


--- New Incoming Records ---
   Student_ID   Name  Age  Marks Performance
0           9  Vikas   21     87   Excellent
1          10  Pooja   22     91   Excellent

--- Records Selected for Incremental Loading ---
   Student_ID   Name  Age  Marks Performance
0           9  Vikas   21     87   Excellent
1          10  Pooja   22     91   Excellent

Incremental loading completed successfully.


STEP 15: FINAL DATABASE CHECK

In [ ]:
print("\n--- Final Students Table ---")

final_data = pd.read_sql(
    "SELECT * FROM Students",
    connection
)

print(final_data)


# ============================================================
# CLOSE DATABASE
# ============================================================

connection.close()

print("\nETL Pipeline completed successfully!")


--- Final Students Table ---
   Student_ID   Name  Age  Marks        Performance
0           1  RAHUL   20     85          Excellent
1           2  SNEHA   21     92          Excellent
2           3   AMIT   22     78               Good
3           4  PRIYA   20     88          Excellent
4           5  ROHAN   22     65  Needs Improvement
5           6   NEHA   21     95          Excellent
6           7  KARAN   20     82               Good
7           8  ANITA   23     90          Excellent
8           9  Vikas   21     87          Excellent
9          10  Pooja   22     91          Excellent

ETL Pipeline completed successfully!
